[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/capture-hlo.ipynb)

# HLO capture day · before and after the compiler

**Hardware:** any Colab TPU runtime. Run the first cell; if it installs anything, Runtime → Restart session, then Run all.

Three small programs captured at two moments: the StableHLO XLA receives, and the optimized HLO it decided on, with its fusion ops. The pair feeds the chapter 04 fusion x-ray so readers can hover a line on one side and see what the compiler did with it. The final cell prints one JSON blob.


In [ ]:
!pip install -q -U "jax[tpu]"
import importlib.metadata as md
print("jax", md.version("jax"), "· libtpu", md.version("libtpu"))


In [ ]:
import os
os.environ.pop("TPU_LIBRARY_PATH", None)
import json, re
import jax
import jax.numpy as jnp

print(jax.__version__, jax.devices())
assert jax.devices()[0].platform == "tpu", "needs a TPU runtime: the optimized dump is backend-specific"
CHIP = jax.devices()[0].device_kind

def naive_attention(q, k, v):
    s = q @ k.T
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    return (p / jnp.sum(p, axis=-1, keepdims=True)) @ v

def gelu_mlp(x, w1, b1, w2):
    return jax.nn.gelu(x @ w1 + b1) @ w2

def row_softmax(x):
    return jax.nn.softmax(x, axis=-1)

S = jax.ShapeDtypeStruct
B = jnp.bfloat16
PROGRAMS = [
    ("attention", "naive attention, seq 1024", "find the fusion ops that carry bf16[1024,1024]: the spill, in the compiler's own plan",
     naive_attention, [S((1024, 128), B)] * 3),
    ("mlp", "matmul, bias, GELU, matmul", "the elementwise chain between the two dots fuses; the dots do not",
     gelu_mlp, [S((256, 512), B), S((512, 2048), B), S((2048,), B), S((2048, 512), B)]),
    ("softmax", "row softmax, 1024x512", "reductions become fusion roots: max and sum each anchor one",
     row_softmax, [S((1024, 512), B)]),
]

def sanitize(text):
    # dumps carry source paths from this runtime; the site never shows them
    text = re.sub(r'source_file="[^"]*"', 'source_file="capture-hlo.ipynb"', text)
    return re.sub(r'"/tmp/[^"]*"', '"capture-hlo.ipynb"', text)

out = {"chip": CHIP, "jax": jax.__version__, "programs": []}
for pid, title, note, fn, args in PROGRAMS:
    lowered = jax.jit(fn).lower(*args)
    unopt = sanitize(lowered.as_text())
    opt = sanitize(lowered.compile().as_text())
    out["programs"].append({"id": pid, "title": title, "note": note,
                            "unopt": unopt.splitlines(), "opt": opt.splitlines()})
    print(f"{pid}: {len(unopt.splitlines())} stablehlo lines -> {len(opt.splitlines())} optimized hlo lines, "
          f"{sum(1 for ln in opt.splitlines() if ' fusion(' in ln or ' = fusion' in ln or 'fusion.' in ln)} fusion mentions")


## The blob


In [ ]:
print("=" * 60)
print("HLO CAPTURE RESULTS · tell the assistant the run is done")
print("=" * 60)
print(json.dumps(out)[:200], "...")
with open("hlo-pairs.json", "w") as f:
    json.dump(out, f, indent=1)
print("full blob written to hlo-pairs.json in the runtime; also printed below in full for tab reading")
print(json.dumps(out, indent=1))
